# FM Radio: RTL-SDR → FPGA FM Demod → AXI-Stream → Audio Codec (ADAU1761)

เล่นเสียง FM ออกหูฟัง (3.5 mm) ของ PYNQ-Z2 โดยให้ FPGA ทำ FM demodulation และส่งเสียงเข้า codec ทาง AXI-Stream ตรง

```
RTL-SDR (USB) → PS (numpy) → AXI DMA MM2S → [FIR → FM demod → LPF1 → LPF2] (PL, 1.92 MSPS → 48 kHz)
                                                        │
                                              axis_broadcaster_0
                                  ├─ M01_AXIS → audio_codec_ctrl_0 (FIFO 16k + I2S) → ADAU1761 → หูฟัง
                                  └─ M00_AXIS → AXI DMA S2MM → output_buffer (ใช้ดู level)
```

| ลำดับ | ส่วน | ทำอะไร |
|---|---|---|
| 1 | Parameters | ตั้งสถานี, volume, จำนวนบล็อก |
| 2 | Load overlay | โหลด `base.bit` และดึง handle ของ IP |
| 3 | Audio codec | configure + unmute ADAU1761 |
| 4 | Test tone (ไม่บังคับ) | ไซน์ 440 Hz จากฮาร์ดแวร์ ใช้เช็กว่า codec ออกเสียงได้ |
| 5 | DMA buffers | จอง input/output buffer |
| 6 | Streaming | อ่าน RTL-SDR แล้วส่งเข้า DMA ต่อเนื่อง |
| 7 | Cleanup | หยุด SDR, คืน buffer |

## 1. Parameters

In [ ]:
# ---- station / RTL-SDR ----
FC   = 99_750_000        # center frequency (Hz)
FS   = 1_920_000         # RTL-SDR sample rate (complex samples / s)
PPM  = 60                # RTL-SDR frequency correction (ppm)

# ---- audio ----
VOLUME   = 60            # headphone volume 0..63
REG_TONE = 0x20          # audio_codec_ctrl_0: 1 = 440 Hz test tone, 0 = AXI-Stream audio

# ---- DMA block (must match the FPGA decimation: 1.92 MSPS -> 48 kHz = /40) ----
DECIM     = 40
BLOCK_IN  = 1024 * 1500            # IQ samples per DMA block (0.8 s)
BLOCK_OUT = BLOCK_IN // DECIM      # audio samples per block (S2MM length)

NUM_BLOCKS = 10                    # blocks to play (10 x 0.8 s = 8 s); None = until the kernel is interrupted

## 2. Load overlay

In [ ]:
import os, fcntl, time, asyncio
import numpy as np
import nest_asyncio
from pynq import Overlay, allocate
from rtlsdr import RtlSdr

print('[1] Loading Overlay...')
ol     = Overlay('./base.bit')
pAudio = ol.audio_codec_ctrl_0
dma    = ol.rtl_sdr.axi_dma_0

# IP contained in the bitstream
print(list(ol.ip_dict.keys()))

## 3. Audio codec (ADAU1761)

`pAudio.configure()` ตั้ง PLL/clock ของ codec ส่วน mixer และ headphone ต้อง unmute ผ่าน I2C เอง
(driver เดิมจะ unmute ให้เฉพาะตอนเรียก `play()`)

| Register | Address | ค่า | ความหมาย |
|---|---|---|---|
| R22 | `0x1C` | `0x21` | Playback mixer left: DAC L |
| R24 | `0x1E` | `0x41` | Playback mixer right: DAC R |
| R29 / R30 | `0x23` / `0x24` | `(vol << 2) \| 0x03` | Headphone L/R volume + enable |

In [ ]:
I2C_SLAVE = 0x0703      # ioctl: select the I2C slave address
ADAU1761  = 0x3B        # codec address on /dev/i2c-1


def codec_unmute(volume=VOLUME):
    """Route the DAC to the headphone amplifier and unmute (ADAU1761 over I2C)."""
    vol = ((volume & 0x3F) << 2) | 0x03
    fd = os.open('/dev/i2c-1', os.O_RDWR)
    try:
        fcntl.ioctl(fd, I2C_SLAVE, ADAU1761)
        for reg, val in ((0x1C, 0x21),      # R22 Playback Mixer Left
                         (0x1E, 0x41),      # R24 Playback Mixer Right
                         (0x23, vol),       # R29 Headphone Left Volume
                         (0x24, vol)):      # R30 Headphone Right Volume
            os.write(fd, bytes([0x40, reg, val]))
    finally:
        os.close(fd)


print('[2] Configuring Audio Codec (ADAU1761)...')
pAudio.mmio.write(REG_TONE, 0)             # audio comes from the AXI-Stream
pAudio.configure(sample_rate=48000, iic_index=1, uio_name='audio-codec-ctrl')
codec_unmute()
print('    ADAU1761 Mixer & Headphone UNMUTED! (volume %d/63)' % VOLUME)

## 4. Test tone (ไม่บังคับ)

ไซน์ 440 Hz สร้างในฮาร์ดแวร์ (`audio_codec_ctrl_0` register `0x20`) ไม่ผ่าน DMA/RTL-SDR
ถ้าได้ยินเสียงนี้ แปลว่า codec และ I2S ใช้ได้

In [ ]:
def tone_test(seconds=2.0):
    print('[3] Testing 440 Hz tone from FPGA hardware for %.1f seconds...' % seconds)
    pAudio.mmio.write(REG_TONE, 1)
    time.sleep(seconds)
    pAudio.mmio.write(REG_TONE, 0)
    print('    Tone finished!')


tone_test()

## 5. DMA buffers

- `input_buffers` (x2): IQ จาก RTL-SDR, 1 sample = 2 byte (I ใน byte ต่ำ, Q ใน byte สูง) สลับใช้ทีละอัน
  เพื่อไม่ให้เขียนทับบัฟเฟอร์ที่ DMA ยังอ่านอยู่
- `output_buffer`: ปลายทางของ DMA S2MM (สำเนาเสียงจาก `M00_AXIS`)
  broadcaster ต้องมีปลายทางพร้อมรับทั้งสองทาง ถ้าไม่มี DMA รับ เสียงที่ไป codec (`M01_AXIS`) ก็จะไม่ไหล

In [ ]:
input_buffers = [allocate(shape=(BLOCK_IN,), dtype=np.uint16) for _ in range(2)]
output_buffer = allocate(shape=(BLOCK_OUT,), dtype=np.int16)

for i, b in enumerate(input_buffers):
    print(f'Input Buffer {i} Address: {hex(b.device_address)}')
print(f'Output Buffer Address: {hex(output_buffer.device_address)}')

## 6. Streaming

วน 1 รอบ = 1 บล็อก (0.8 s):
1. รับ IQ 1,536,000 ตัวอย่างจาก RTL-SDR
2. แปลง unsigned 8-bit (กลาง 128) เป็น signed 8-bit แล้ว pack I/Q เป็น `uint16`
3. รอ DMA ของบล็อกก่อนหน้าให้จบ แล้วส่งบล็อกใหม่ (MM2S) พร้อมเปิดรับเสียง (S2MM)

FIFO ใน `audio_codec_ctrl_0` (16k ตัวอย่าง ≈ 0.34 s) เก็บเสียงคั่นช่วงระหว่างบล็อก

In [ ]:
nest_asyncio.apply()                 # allow asyncio.run() inside Jupyter


def open_sdr():
    sdr = RtlSdr()
    sdr.sample_rate     = FS
    sdr.center_freq     = FC
    sdr.freq_correction = PPM
    sdr.gain            = 'auto'
    return sdr


def wait_dma():
    dma.sendchannel.wait()
    dma.recvchannel.wait()


async def stream_fm(sdr, num_blocks):
    idx    = 0           # which input buffer is filled next
    active = False       # a DMA transfer is in flight
    blocks = 0
    try:
        async for data in sdr.stream(num_samples_or_bytes=BLOCK_IN * 2, format='bytes', loop=None):
            # uint8 (centre 128) -> int8 is x ^ 0x80; I = low byte, Q = high byte of each uint16
            packed = (np.frombuffer(data, dtype=np.uint8) ^ 0x80).view(np.uint16)
            np.copyto(input_buffers[idx], packed)

            level = 0
            if active:                                   # previous block must be finished first
                wait_dma()
                level = int(np.abs(output_buffer.astype(np.int32)).max())

            dma.recvchannel.transfer(output_buffer)      # S2MM first: M00_AXIS must never block
            dma.sendchannel.transfer(input_buffers[idx])
            active = True
            idx ^= 1

            blocks += 1
            print(f'block {blocks}' + (f'/{num_blocks}' if num_blocks else '') + f'  level {level}')
            if num_blocks and blocks >= num_blocks:
                print('Stopping stream...')
                break
    finally:
        try:
            await sdr.stop()
        except Exception as e:                           # noqa: BLE001
            print('sdr.stop():', e)
        if active:
            wait_dma()                                   # let the last block finish


def run_fm(num_blocks=NUM_BLOCKS):
    pAudio.mmio.write(REG_TONE, 0)                       # audio from the AXI-Stream, not the test tone
    sdr = open_sdr()
    os.system('echo 0 > /sys/module/usbcore/parameters/usbfs_memory_mb')   # unlimited USB buffer
    try:
        asyncio.run(stream_fm(sdr, num_blocks))
    except KeyboardInterrupt:
        print('Interrupted')
    finally:
        sdr.close()

In [ ]:
run_fm()

## 7. Cleanup

คืน DMA buffer เมื่อเลิกใช้ (ถ้าจะรันสตรีมต่อ ให้ข้ามเซลล์นี้ แล้วเรียก `run_fm()` อีกครั้งได้เลย)

In [ ]:
pAudio.mmio.write(REG_TONE, 0)
for b in input_buffers:
    b.freebuffer()
output_buffer.freebuffer()
del input_buffers, output_buffer

### ปัญหาที่พบบ่อย

| อาการ | สาเหตุที่เป็นไปได้ |
|---|---|
| ไม่มีเสียงเลย แม้ Test tone | codec ยังไม่ unmute (รันเซลล์ section 3) หรือเสียบหูฟังผิดช่อง |
| Test tone มีเสียง แต่ FM เงียบ, `level 0` | RTL-SDR ไม่ได้ข้อมูล / จูนผิดสถานี / DMA ไม่ได้ส่ง |
| `level` ขึ้นแต่เสียงขาดเป็นช่วงๆ | ช่องว่างระหว่างบล็อกยาวเกินกว่า FIFO (0.34 s) รองรับ |
| `RuntimeError: DMA channel not idle` | รันบล็อกซ้อน: เซลล์ `run_fm()` ถูกขัดจังหวะกลางทาง ให้รัน Section 2 ใหม่ |